In [1]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path.cwd() / 'data'
if not DATA_DIR.exists():
    DATA_DIR = Path.cwd().parent / 'data'

In [2]:
full_merged_data = pd.read_csv(DATA_DIR / 'merged_geo.csv')

In [3]:
# College attribute coverage (non-empty %) across the merged network dataset

college_attr_cols = [c for c in full_merged_data.columns if c.startswith('col_')]

print('Rows:', len(full_merged_data))
print('Unique colleges:', full_merged_data['college_id'].nunique())
print('College attribute columns:', len(college_attr_cols))
college_attr_cols

Rows: 1048575
Unique colleges: 2846
College attribute columns: 13


['col_name',
 'col_city',
 'col_st',
 'col_zip',
 'col_type',
 'col_ctyname',
 'col_ctyfips',
 'col_shparea',
 'col_shplength',
 'col_inst_control',
 'col_inst_size',
 'col_endow_total',
 'col_endow_per_fte']

In [4]:
def _non_empty_mask(s: pd.Series) -> pd.Series:
    if pd.api.types.is_string_dtype(s) or s.dtype == object:
        s_str = s.astype('string').str.strip()
        return s_str.notna() & (s_str != '')
    return s.notna()

summary_rows = []
n = len(full_merged_data)

for c in college_attr_cols:
    m = _non_empty_mask(full_merged_data[c])
    non_empty = int(m.sum())
    summary_rows.append({
        'field': c,
        'non_empty_cells': non_empty,
        'total_cells': n,
        'pct_non_empty': (non_empty / n * 100.0) if n else 0.0,
    })

college_attr_coverage = (
    pd.DataFrame(summary_rows)
      .sort_values('pct_non_empty', ascending=False)
      .reset_index(drop=True)
)
college_attr_coverage

,field,non_empty_cells,total_cells,pct_non_empty
0,col_name,1020156,1048575,97.289750
1,col_city,1020156,1048575,97.289750
2,col_st,1020156,1048575,97.289750
3,col_zip,1020156,1048575,97.289750
4,col_type,1020156,1048575,97.289750
5,col_ctyname,1020156,1048575,97.289750
6,col_ctyfips,1020156,1048575,97.289750
7,col_shparea,1020156,1048575,97.289750
8,col_shplength,1020156,1048575,97.289750
9,col_inst_control,885783,1048575,84.474930


In [5]:
# Colleges with NO data in any college attribute field (all col_* empty)

if not college_attr_cols:
    colleges_no_col_data = pd.DataFrame(columns=['college_id', 'n_rows', 'n_cycles'])
else:
    any_non_empty_by_row = pd.Series(False, index=full_merged_data.index)
    for c in college_attr_cols:
        any_non_empty_by_row |= _non_empty_mask(full_merged_data[c])

    any_non_empty_by_college = any_non_empty_by_row.groupby(full_merged_data['college_id']).any()
    missing_college_ids = any_non_empty_by_college[~any_non_empty_by_college].index

    colleges_no_col_data = (
        full_merged_data.loc[full_merged_data['college_id'].isin(missing_college_ids), ['college_id', 'cycle']]
          .groupby('college_id', as_index=False)
          .agg(n_rows=('college_id', 'size'), n_cycles=('cycle', 'nunique'))
          .sort_values(['n_rows', 'college_id'], ascending=[False, True])
          .reset_index(drop=True)
    )

print('Colleges with zero college-attribute matches:', len(colleges_no_col_data))
colleges_no_col_data.head()

Colleges with zero college-attribute matches: 140


,college_id,n_rows,n_cycles
0,192439,1867,5
1,145770,1517,5
2,492041,954,5
3,441308,824,5
4,208239,680,5
